In [ ]:
from torchvision import models
from torchvision import transforms
from PIL import Image 
import torch
import urllib 

In [6]:
dir(models)
alexnet = models.AlexNet()
resnet = models.resnet101(pretrained=True)

/home/olena/miniconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/olena/miniconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
preprocess = transforms.Compose([  #конвейер для предобработки изображения
    transforms.Resize(256), #прорционально уменьшить размер изображения, меньшая сторона - 256 пикселей
    transforms.CenterCrop(224), #из уменьшенного изображения вырезать маленький квадрат 224х224 (стандарт)
    transforms.ToTensor(), #изображени превращается в массив чисел (3, 224, 224) + числа переводятся в диапазон [0, 1]
    transforms.Normalize( #изменяет значения так, что среднее 0, а разброс 1, тоесть диапазон [-2, 2]
        mean=[0.485, 0.456, 0.406], #для красного канала еозначает, что в среднем картинки в ImageNet заполнены красным цветом примерно на 48.5%
        std=[0.229, 0.224, 0.225] #показывает, насколько сильно значения пикселей «разбросаны» относительно среднего
    )
])

In [8]:
img = Image.open('/home/olena/Зображення/bobby.jpg')
img_t = preprocess(img)
batch_t = torch.unsqueeze(img_t, 0) #размер (1, 3, 224, 224), где 1 - батч из 1 изображения

In [ ]:
resnet.eval() #режим тестирования
out = resnet(batch_t)

In [10]:
file_path = '/home/olena/Git_repos/learning-pytorch/imnet_classes.txt'
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r') as f:
    labels = []
    for line in f.readlines():
        cleaned_line = line.strip()
        labels.append(cleaned_line)

probabilities = torch.nn.functional.softmax(out, dim=1)[0] #превращение цифр в вероятности
top_prob, top_index = torch.max(probabilities, 0)
class_id = top_index.item()
predicted_label = labels[class_id]

In [11]:
predicted_label  

'golden retriever'